# 07_agent_debugging_and_failures: Infinite loops, Tool exception recovery

This notebook maps standard agent failure states (such as infinite execution loop loops, tool API exceptions, and context memory saturation) and implements recovery handlers for each.

### Failure Scenarios
1. **Infinite loops**: Agent repeats the same tool call. Prevented via recursion state monitors.
2. **Tool Exceptions**: Simulated rate limits (HTTP 429). Recovered via fallback logic.
3. **Context Saturation**: Context size limits. Resolved by sliding window trimming.

In [1]:
# 1. Infinite Loop Monitor Implementation
class LoopMonitorException(Exception):
    pass

def execute_agent_loop(query: str, max_iterations: int = 3):
    state_history = []
    print("Goal:", query)
    for i in range(5):
        try:
            action = "search_web('agent')"
            print(f"Step {i+1}: Action = {action}")
            if action in state_history:
                raise LoopMonitorException("Infinite loop detected: repeat tool call action!")
            state_history.append(action)
        except LoopMonitorException as e:
            print(f"[RECOVERY]: Caught exception: {e}")
            print("[RECOVERY]: Terminated execution and returned fallback resolution.")
            break

execute_agent_loop("Find AI agent definitions.")

Goal: Find AI agent definitions.
Step 1: Action = search_web('agent')
Step 2: Action = search_web('agent')
[RECOVERY]: Caught exception: Infinite loop detected: repeat tool call action!
[RECOVERY]: Terminated execution and returned fallback resolution.


In [2]:
# 2. Tool failure fallback logic
def fetch_data_from_tool():
    try:
        raise ConnectionError("Rate limit exceeded (HTTP 429).")
    except ConnectionError as e:
        print(f"[RECOVERY]: Caught tool error: {e}")
        print("[RECOVERY]: Falling back to local cache database.")
        return "Local Cache Profile Data (Fallback)"

result = fetch_data_from_tool()
print("Tool response:", result)

[RECOVERY]: Caught tool error: Rate limit exceeded (HTTP 429).
[RECOVERY]: Falling back to local cache database.
Tool response: Local Cache Profile Data (Fallback)


In [3]:
# 3. Context window length trimming
contexts = ["First long dialogue block...", "Second long dialogue block...", "Third long dialogue block..."]
print("Original Context Count:", len(contexts))
trimmed_contexts = contexts[-2:]
print("Trimmed Context Count:", len(trimmed_contexts))

Original Context Count: 3
Trimmed Context Count: 2


### Output Explanation & Recovery Verification

#### Executed Results Trace:
- **Loop Protection**: Detected a duplicate action `search_web('agent')` at Step 2 and triggered the `LoopMonitorException` recovery sequence to prevent infinite credit spend.
- **Tool Recovery**: Successfully caught the connection rate limit error and fell back to returning cached user database results.
- **Context Trimming**: Successfully trimmed context counts from 3 blocks to the last 2, preventing token window saturation.